# Reference-Metric Applications

Use reference metrics to read coordinate maps, scale factors, determinant identities, and precomputed factors.

Navigation: [Index](../index.ipynb) | Previous: [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb) | Next: [Basis Transforms](basis_transforms.ipynb)


## Learning Goals

- Use reference metrics to organize coordinate geometry.
- Check determinant and scale-factor identities.
- See how precomputed geometry data supports generated curvilinear code.

## Words for This Notebook

- **Reference metric:** the built-in geometry of a coordinate system.
- **Jacobian:** a table of derivatives that converts between coordinate labels.
- **Determinant:** a single number built from a matrix that often measures volume scaling.
- **Precompute:** calculate reusable coordinate quantities once instead of repeatedly.

Use the code cells actively: first predict what should happen, then run the cell, then explain the output in plain language. This predict-run-explain pattern keeps the physics idea connected to the programming details.


## Coordinate Maps, Scale Factors, and Precompute Data
The determinant identity checks that the product of scale factors matches the reference-metric determinant in spherical coordinates.

## Import SymPy for Metric Identities

These imports expose the NRPy and Python tools used in the next steps.


In [1]:
import sympy as sp


## Import Reference-Metric Application Tools

These imports expose the NRPy registries and infrastructure writers used below.


In [2]:
import nrpy.params as par
import nrpy.reference_metric as refmetric
from nrpy.infrastructures.BHaH.rfm_precompute import ReferenceMetricPrecompute


## Compare Cartesian and Spherical Metrics

The output connects coordinate maps to metric factors.


In [3]:
for coord_system in ["Cartesian", "Spherical", "SinhSpherical", "SinhCylindrical"]:
    metric = refmetric.ReferenceMetric(coord_system, enable_rfm_precompute=False)
    print()
    print(coord_system)
    print("xx:", metric.xx)
    print("xx_to_Cart:", metric.xx_to_Cart)
    print("scale factors:", metric.scalefactor_orthog)
    print("ghatDD diagonal:", [metric.ghatDD[i][i] for i in range(3)])
    print("ghatUU diagonal:", [metric.ghatUU[i][i] for i in range(3)])
    print("detgammahat:", metric.detgammahat)
    print("radial-like directions:", metric.radial_like_dirns)



Cartesian
xx: [xx0, xx1, xx2]
xx_to_Cart: [xx0, xx1, xx2]
scale factors: [1, 1, 1]
ghatDD diagonal: [1, 1, 1]
ghatUU diagonal: [1, 1, 1]
detgammahat: 1
radial-like directions: [0, 1, 2]

Spherical
xx: [xx0, xx1, xx2]
xx_to_Cart: [xx0*sin(xx1)*cos(xx2), xx0*sin(xx1)*sin(xx2), xx0*cos(xx1)]
scale factors: [1, xx0, xx0*sin(xx1)]
ghatDD diagonal: [1, xx0**2, xx0**2*sin(xx1)**2]
ghatUU diagonal: [1, xx0**(-2), 1/(xx0**2*sin(xx1)**2)]
detgammahat: xx0**4*sin(xx1)**2
radial-like directions: [0]

SinhSpherical
xx: [xx0, xx1, xx2]
xx_to_Cart: [AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*sin(xx1)*cos(xx2)/(exp(1/SINHW) - exp(-1/SINHW)), AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*sin(xx1)*sin(xx2)/(exp(1/SINHW) - exp(-1/SINHW)), AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*cos(xx1)/(exp(1/SINHW) - exp(-1/SINHW))]
scale factors: [AMPL*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SINHW)/(exp(1/SINHW) - exp(-1/SINHW)), AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))/(exp(1/SINHW) - exp(-1/SINHW)), AMPL*(exp(xx0/SINHW) - ex


SinhCylindrical
xx: [xx0, xx1, xx2]
xx_to_Cart: [AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))*cos(xx1)/(exp(1/SINHWRHO) - exp(-1/SINHWRHO)), AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))*sin(xx1)/(exp(1/SINHWRHO) - exp(-1/SINHWRHO)), AMPLZ*(exp(xx2/SINHWZ) - exp(-xx2/SINHWZ))/(exp(1/SINHWZ) - exp(-1/SINHWZ))]
scale factors: [AMPLRHO*(exp(xx0/SINHWRHO)/SINHWRHO + exp(-xx0/SINHWRHO)/SINHWRHO)/(exp(1/SINHWRHO) - exp(-1/SINHWRHO)), AMPLRHO*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))/(exp(1/SINHWRHO) - exp(-1/SINHWRHO)), AMPLZ*(exp(xx2/SINHWZ)/SINHWZ + exp(-xx2/SINHWZ)/SINHWZ)/(exp(1/SINHWZ) - exp(-1/SINHWZ))]
ghatDD diagonal: [AMPLRHO**2*(exp(xx0/SINHWRHO)/SINHWRHO + exp(-xx0/SINHWRHO)/SINHWRHO)**2/(exp(1/SINHWRHO) - exp(-1/SINHWRHO))**2, AMPLRHO**2*(exp(xx0/SINHWRHO) - exp(-xx0/SINHWRHO))**2/(exp(1/SINHWRHO) - exp(-1/SINHWRHO))**2, AMPLZ**2*(exp(xx2/SINHWZ)/SINHWZ + exp(-xx2/SINHWZ)/SINHWZ)**2/(exp(1/SINHWZ) - exp(-1/SINHWZ))**2]
ghatUU diagonal: [(exp(1/SINHWRHO) - exp(-1/SINHWRHO))

## Verify the Spherical Determinant Identity

A zero residual confirms that the symbolic expression matches the expected identity.


In [4]:
spherical = refmetric.ReferenceMetric("Spherical", enable_rfm_precompute=False)
scale_product = sp.prod(spherical.scalefactor_orthog)
print(
    "Spherical determinant residual:",
    sp.trigsimp(sp.simplify(spherical.detgammahat - scale_product**2)),
)


Spherical determinant residual: 0


## Register Precompute Support Data

Named parameters keep numerical choices separate from symbolic formulas.


In [5]:
par.set_parval_from_str("parallelization", "openmp")
par.set_parval_from_str("fp_type", "double")
par.set_parval_from_str("CoordSystem_to_register_CodeParameters", "SinhCylindrical")
precompute_metric = refmetric.reference_metric["SinhCylindrical_rfm_precompute"]
precompute = ReferenceMetricPrecompute("SinhCylindrical")
print("precompute metric:", precompute_metric.CoordSystem)
print("stored member count:", len(precompute.member_specs))
print("member names:")
for name, declaration in precompute.member_specs:
    print(name, declaration)


Setting up reference_metric[SinhCylindrical_rfm_precompute]...
precompute metric: SinhCylindrical
stored member count: 7
member names:
f0_of_xx0 (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__D0 (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__DD00 (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__DDD000 (size_t)params->Nxx_plus_2NGHOSTS0
f3_of_xx2 (size_t)params->Nxx_plus_2NGHOSTS2
f3_of_xx2__D2 (size_t)params->Nxx_plus_2NGHOSTS2
f3_of_xx2__DD22 (size_t)params->Nxx_plus_2NGHOSTS2


The printed Jacobian and metric factors show how coordinate geometry enters a PDE. These factors are what curvilinear wave update routines use to keep the physics independent of the chosen coordinates.


## Learning Check

Before checking the determinant identity, predict why spherical coordinates need a volume factor involving `r` and `sin(theta)`.


## Continue to Basis Transforms
- [Reference Metrics](../1-intro/reference_metric.ipynb)
- [Basis Transforms](basis_transforms.ipynb)
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)
